# Notebook 05 — Data Cleaning and Feature Engineering

## Purpose
I clean all tables based on findings from Notebook 04, then merge them into a
single flat `master` dataframe.

## Why this matters
A clean, merged master table is the single source of truth for all subsequent
analysis and modelling.  All preprocessing decisions are documented here.

## Inputs
`data/raw/` — all 9 CSV files

## Outputs
`data/processed/master.parquet` — the analysis-ready master table

## Key decisions
- Keep delivered orders only (others lack review/delivery data)
- Drop orders with order_value <= 0
- Fill missing product weights with the category median


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path('..').resolve()))
from src.config import load_config
from src.paths import Paths
from src.preprocessing import (
    parse_dates, compute_delivery_features,
    aggregate_order_items, aggregate_payments,
    aggregate_reviews, merge_product_features,
    build_master, check_schema,
)
from src.utils import set_all_seeds, save_table

cfg   = load_config()
paths = Paths(cfg)
SEED  = cfg['project']['random_seed']
set_all_seeds(SEED)

print("Imports OK. Loading raw CSVs...")


  Random seed set to 42
Imports OK. Loading raw CSVs...


In [2]:
# Load all tables
DATE_COLS = cfg['preprocessing']['date_columns']

orders      = pd.read_csv(paths.olist_file('orders', cfg))
items       = pd.read_csv(paths.olist_file('order_items', cfg))
payments    = pd.read_csv(paths.olist_file('order_payments', cfg))
reviews     = pd.read_csv(paths.olist_file('order_reviews', cfg))
customers   = pd.read_csv(paths.olist_file('customers', cfg))
products    = pd.read_csv(paths.olist_file('products', cfg))
sellers     = pd.read_csv(paths.olist_file('sellers', cfg))
translation = pd.read_csv(paths.olist_file('category_translation', cfg))

print(f"Raw orders: {len(orders):,}")


Raw orders: 99,441


In [3]:
# Step 1: Parse dates
print("Step 1: Parsing date columns...")
orders = parse_dates(orders, DATE_COLS)
print(f"  Date range: {orders['order_purchase_timestamp'].min().date()} "
      f"to {orders['order_purchase_timestamp'].max().date()}")


Step 1: Parsing date columns...
  Date range: 2016-09-04 to 2018-10-17


In [4]:
# Step 2: Fix product weight NaN by filling with category median
print("Step 2: Filling missing product weights with category median...")
products['product_weight_g'] = products.groupby('product_category_name')[
    'product_weight_g'
].transform(lambda x: x.fillna(x.median()))
# Remaining NaN (categories where all weights are null): fill with overall median
overall_median_weight = products['product_weight_g'].median()
products['product_weight_g'] = products['product_weight_g'].fillna(overall_median_weight)
print(f"  Remaining weight nulls: {products['product_weight_g'].isna().sum()}")


Step 2: Filling missing product weights with category median...
  Remaining weight nulls: 0


In [5]:
# Step 3: Build master — this calls all merge and aggregation logic from src/preprocessing.py
print("Step 3: Building master table...")
print()
master = build_master(
    orders, items, payments, reviews,
    customers, products, sellers, translation,
    cfg=cfg,
)
print()
print("Master table shape:", master.shape)
print("Columns:", list(master.columns))


Step 3: Building master table...

  Delivered orders: 96,478 / 99,441 total
  Master shape: (96478, 33)
  Date range: 2016-09-15 → 2018-08-29

Master table shape: (96478, 33)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days', 'is_late', 'days_to_deliver', 'days_to_approve', 'order_value', 'freight_value', 'n_items', 'freight_ratio', 'payment_type', 'payment_value', 'payment_installments', 'review_score', 'product_category_english', 'product_weight_g', 'product_photos_qty', 'seller_state', 'customer_unique_id', 'customer_state', 'purchase_month', 'purchase_dayofweek', 'purchase_year', 'customer_state_encoded', 'seller_state_encoded', 'product_category_encoded', 'payment_type_encoded']


In [6]:
# Step 4: Check key derived columns
print("=== Derived column verification ===")
print()
print("delivery_delay_days sample:")
print(master['delivery_delay_days'].describe())
print()
print("is_late value counts:")
print(master['is_late'].value_counts())
print()
print("review_score value counts:")
print(master['review_score'].value_counts().sort_index())


=== Derived column verification ===

delivery_delay_days sample:
count    96470.000000
mean       -11.875889
std         10.182105
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

is_late value counts:
is_late
0.0    89936
1.0     6534
Name: count, dtype: int64

review_score value counts:
review_score
1.0     9352
2.0     2920
3.0     7916
4.0    18894
5.0    56750
Name: count, dtype: int64


In [7]:
# Step 5: Final null check on master
from src.utils import missing_summary
ms = missing_summary(master)
if len(ms) > 0:
    print("Remaining missing values in master:")
    print(ms.to_string(index=False))
else:
    print("No missing values in key analytical columns.")

# I accept NaN in review_score for orders where the customer did not review
pct_no_review = master['review_score'].isna().mean() * 100
print(f"\nOrders with no review score: {pct_no_review:.1f}% (acceptable - not all orders are reviewed)")


Remaining missing values in master:
                       column  n_missing  pct_missing
           product_photos_qty       1359         1.41
                 review_score        646         0.67
            order_approved_at         14         0.01
order_delivered_customer_date          8         0.01
              days_to_approve         14         0.01
          delivery_delay_days          8         0.01
                      is_late          8         0.01
              days_to_deliver          8         0.01
 order_delivered_carrier_date          2         0.00
                 payment_type          1         0.00
         payment_installments          1         0.00
                payment_value          1         0.00

Orders with no review score: 0.7% (acceptable - not all orders are reviewed)


In [8]:
# Step 6: Save master to processed/
paths.processed.mkdir(parents=True, exist_ok=True)
master_path = paths.processed / cfg['data']['master_file']
master.to_parquet(master_path, index=False)
print(f"Saved master to: {master_path}")
print(f"Shape: {master.shape}")
print()

# Quick summary
print("=== Master table summary ===")
print(f"Total delivered orders: {len(master):,}")
print(f"Date range: {master['order_purchase_timestamp'].min().date()} to "
      f"{master['order_purchase_timestamp'].max().date()}")
print(f"Unique customers: {master['customer_unique_id'].nunique():,}")
print(f"Total revenue: R$ {master['order_value'].sum():,.2f}")
print(f"Late delivery rate: {master['is_late'].mean()*100:.1f}%")
print(f"Avg review score: {master['review_score'].mean():.3f}")


Saved master to: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\data\processed\master.parquet
Shape: (96478, 33)

=== Master table summary ===
Total delivered orders: 96,478
Date range: 2016-09-15 to 2018-08-29
Unique customers: 93,358
Total revenue: R$ 13,221,498.11
Late delivery rate: 6.8%
Avg review score: 4.156
